# 06 · Attenuation of X-rays: thickness and material

EPN Práctica 1 on the 554 800: the GM counter sits in the direct beam (no crystal), and absorbers from the accessory
554 834 go in front of it. Lambert's law ln T = −μ d with T = (R − b)/(R₀ − b) gives the linear attenuation
coefficient μ from the slope of ln T against thickness d, and μ/ρ = 10 μ/ρ(g/cm³) in cm²/g. The analysis lives in
`braggsim.analysis.attenuation_fit`; this notebook runs it and shows what the guide's students will see, including the
two things that make the line not straight: the polychromatic beam (beam hardening) and the counter's dead time.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xraylib

from braggsim.analysis import attenuation_fit, scaled_transmission
from braggsim.constants import HC_KEV_PM, MO_KA_PM
from braggsim.detector import dead_time, dead_time_correct, sample_counts
from braggsim.filters import WHEELS_MM, areal_density_mg_cm2
from braggsim.scan import DEFAULT, direct_rate

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
DATA, MODEL, ALT = "#1a1a19", "#2a78d6", "#eb6834"

## The guide's settings

| Run | Setting |
|---|---|
| 1a | Al wheel, 0.5 → 3.0 mm, 21 kV, 0.05 mA, 3 × 30 s per position |
| 1b | the same with the collimator's Zr filter, 21 kV, 0.15 mA, 200 s |
| (extra, not in the guide) | 35 kV with Zr, 0.01 mA (keeps the counter below ≈ 1000/s; on a real tube the emission may drift at this current, so read R₀ before and after), 200 s |
| 2a | material plates (0.4 mm), 30 kV, 0.02 mA for polystyrene / Al / Fe, 1 mA for Cu / Zr / Ag |
| 3 | background, 0 kV, ≥ 100 s |

Poisson noise is the only random step (fixed seed). A reading is the counts of one gate divided by its length.

In [ ]:
rng = np.random.default_rng(2026)
ZR = (("Zr", areal_density_mg_cm2("Zr", 0.05)),)
AL_MM = np.array([mm for _, mm in WHEELS_MM["al"][1:]])
RHO_AL = areal_density_mg_cm2("Al", 1.0) / 100  # g/cm³


def read(U_kV, I_mA, filters, dt_s, params=DEFAULT):
    return sample_counts(direct_rate(U_kV, I_mA, filters, params), dt_s, rng) / dt_s


def al_wheel(U_kV, I_mA, dt_s, filters=(), params=DEFAULT):
    r0 = read(U_kV, I_mA, filters, dt_s, params)
    r = np.array(
        [
            read(U_kV, I_mA, (*filters, ("Al", areal_density_mg_cm2("Al", mm))), dt_s, params)
            for mm in AL_MM
        ]
    )
    return r0, r


BG_DT_S = 100.0
bg = read(0.0, 0.0, (), BG_DT_S)
print(f"background {bg:.2f} /s")
runs = {
    "1a  21 kV, no Zr": (21.0, 0.05, 90.0, ()),
    "1b  21 kV, Zr": (21.0, 0.15, 200.0, ZR),
    "    35 kV, Zr (not in guide)": (35.0, 0.01, 200.0, ZR),
}
data, fits = {}, {}
for name, (U, I_mA, dt, f) in runs.items():
    r0, r = al_wheel(U, I_mA, dt, f)
    data[name] = (r0, r, dt)
    fits[name] = attenuation_fit(AL_MM, r, r0, dt, dt, bg, BG_DT_S)
pd.DataFrame(
    {
        "R0 (1/s)": [f"{data[n][0]:.1f}" for n in runs],
        "mu (1/mm)": [f"{fits[n].mu_per_mm:.3f} ± {fits[n].mu_err_per_mm:.3f}" for n in runs],
        "mu/rho (cm2/g)": [f"{10 * fits[n].mu_per_mm / RHO_AL:.2f}" for n in runs],
        "chi2/dof": [f"{fits[n].chi2_dof:.0f}" for n in runs],
    },
    index=list(runs),
)  # fmt: skip

## T against thickness and the line of ln T

The fitted line goes through the origin (T = 1 at d = 0). Its χ²/dof is far above 1 without the Zr filter: the points
are not on a line, and the ± is the counting error of a line that does not exist. The error bars of the individual
points include the common R₀ and background error.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for (name, f), color in zip(fits.items(), (DATA, MODEL, ALT), strict=True):
    axes[0].errorbar(AL_MM, f.T, f.T_err, fmt="o", ms=4, color=color, label=name.strip())
    axes[1].errorbar(AL_MM, np.log(f.T), f.T_err / f.T, fmt="o", ms=4, color=color)
    axes[1].plot([0, 3.1], [0, -f.mu_per_mm * 3.1], "--", lw=1, color=color)
axes[0].set(xlabel="d (mm Al)", ylabel="T")
axes[1].set(xlabel="d (mm Al)", ylabel="ln T")
axes[0].legend(frameon=False)
fig.tight_layout()

## μ_eff per point: beam hardening

μ_eff,i = −ln T_i / d_i is the slope from the origin to point i. For a monochromatic beam it is constant. Here it falls
with thickness because thin Al removes the softest photons first and what remains is harder (beam hardening).
At 21 kV the Mo K lines are barely excited (≈ 5 % of the photons): the beam is bremsstrahlung from λ_min = 59 pm,
centred near 76 pm, softer than Mo Kα, so the fit lies above the Kα value (5.11 cm²/g) and no run reaches it. The Zr
filter does not make the beam Mo Kα: at 21 kV it removes the hard end (above the Zr K edge) and leaves a narrower
69–92 pm band, so the line is somewhat less curved but μ_eff is higher. At 35 kV the lines are strong (≈ 40 %), but the
continuum above the Zr edge leaks through and survives the Al best, so μ/ρ is below the Kα value and keeps falling.
The dashed line is xraylib's μ/ρ of Al at Mo Kα, the number a student looks up.

In [ ]:
mu_ka = xraylib.CS_Total_CP("Al", HC_KEV_PM / MO_KA_PM)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for (name, f), color in zip(fits.items(), (DATA, MODEL, ALT), strict=True):
    ax.plot(AL_MM, 10 * f.mu_eff_per_mm / RHO_AL, "o-", ms=4, color=color, label=name.strip())
ax.axhline(mu_ka, color="#999", ls="--", lw=1, label=f"xraylib, Mo Kα: {mu_ka:.2f}")
ax.set(xlabel="d (mm Al)", ylabel="μ_eff/ρ (cm²/g)")
ax.legend(frameon=False)
fig.tight_layout()

## Material plates and μ/ρ against Z

Run 2a: 0.4 mm plates at 30 kV. The light plates are read at 0.02 mA and the heavy ones at 1 mA; the open beam is
read at 0.02 mA. `scaled_transmission` corrects every rate for dead time (τ = 100 µs: the open beam at 1 mA would
paralyse the counter), subtracts the background and scales the open beam by the current (×50). μ/ρ = −ln T/(ρx), one
point per plate. The error includes the dead-time Jacobian; a real dead-time-limited GM is sub-Poissonian
(≈ N(1−Rτ)²), which roughly cancels it, while the simulation draws plain Poisson counts.

In [ ]:
PLATES = [m for m, _ in WHEELS_MM["material"][:6]]
Z = {"Polystyrene": 6, "Al": 13, "Fe": 26, "Cu": 29, "Zr": 40, "Ag": 47}  # 6 = C-like
CURRENT = {"Polystyrene": 0.02, "Al": 0.02, "Fe": 0.02, "Cu": 1.0, "Zr": 1.0, "Ag": 1.0}
GATE_S = 90.0
tau = DEFAULT.tau_s
open_obs = read(30.0, 0.02, (), GATE_S)
rows = []
for m in PLATES:
    ad = areal_density_mg_cm2(m, 0.4)
    i = CURRENT[m]
    obs = read(30.0, i, ((m, ad),), GATE_S)
    T, T_err = scaled_transmission(obs, open_obs, i, 0.02, GATE_S, GATE_S, bg, BG_DT_S, tau)
    mu_rho = -np.log(T) / (ad / 1000)
    rows.append((m, i, obs, T, mu_rho, T_err / T / (ad / 1000)))
# xraylib monochromatic reference at Mo Kα
ref = {m: xraylib.CS_Total_CP(m, HC_KEV_PM / MO_KA_PM) for m in PLATES}
tab = pd.DataFrame(rows, columns=["plate", "I (mA)", "R (1/s)", "T", "mu/rho", "err"])
tab["xraylib Mo Ka"] = [ref[m] for m in tab.plate]
tab.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.8))
zz = [Z[m] for m in tab.plate]
ax.errorbar(zz, tab["mu/rho"], tab.err, fmt="o", color=DATA, ms=5, label="simulated, 30 kV")
ax.plot(zz, tab["xraylib Mo Ka"], "s--", color=ALT, ms=5, label="xraylib, Mo Kα")
ax.annotate("Zr", (40, tab["mu/rho"][4]), textcoords="offset points", xytext=(6, 6))
ax.set(xscale="log", yscale="log", xlabel="Z", ylabel="μ/ρ (cm²/g)")
ax.legend(frameon=False)
fig.tight_layout()

Zr is the anomaly. In the monochromatic column μ/ρ drops from Cu to Zr because Mo Kα (17.5 keV) lies below the Zr K
edge (18.0 keV). In the measurement the Zr plate transmits more than Fe or Cu because Mo Kα falls in Zr's low-absorption
window just below its edge (≈ 70 % of what passes is Mo Kα), while Fe and Cu pass only the hard end of the continuum
(≈ 23–29 keV). Fe, Cu and Ag read far below their Mo Kα values for that reason: Ag filters the beam itself (K edge at
25.5 keV), so the survivors sit just below it. Kβ (19.6 keV) is above the Zr edge and is absorbed. μ/ρ against Z is
not monotonic across an edge (`model/PARAMETERS.md`, "Material plate readings").

## Sensitivity to the Pyrex wall

The tube window is modelled as a Pyrex absorber, `absorber_mg_cm2` in `ModelParams`, fitted to the leaflet Fig. 4 and
uncertain at the 20 % level. How much does the Al μ_eff (21 kV, no Zr, noise-free) move if that is wrong?

In [ ]:
def mu_eff_rho(params):
    r = np.array(
        [direct_rate(21.0, 0.05, (("Al", areal_density_mg_cm2("Al", mm)),), params) for mm in AL_MM]
    )
    r0 = direct_rate(21.0, 0.05, (), params)
    fit = attenuation_fit(AL_MM, r, r0, 1.0)
    return 10 * fit.mu_eff_per_mm / RHO_AL


true = DEFAULT.replace(tau_s=0.0, background_per_s=0.0)
base = mu_eff_rho(true)
out = {}
for k in (0.8, 1.0, 1.2):
    p = true.replace(absorber_mg_cm2=true.absorber_mg_cm2 * k)
    out[f"Pyrex x{k}"] = mu_eff_rho(p)
sens = pd.DataFrame(out, index=[f"{mm:.1f} mm" for mm in AL_MM]).T
display(sens.round(2))
print("change at 0.5 mm / 3 mm (%):")
for k in (0.8, 1.2):
    print(k, [f"{100 * (out[f'Pyrex x{k}'][j] / base[j] - 1):+.1f}" for j in (0, -1)])

## Dead time

At the unattenuated ≈ 950/s the GM (non-paralysable, τ = 100 µs) loses about 9 % of the counts, and the loss
shrinks as the absorber lowers the rate. The observed ln T therefore bends: the fitted μ/ρ comes out ≈ 4 % low and μ_eff at 0.5 mm 6–7 % low, and part of the curvature is hidden. Left: the observed
rate against the true one; right: the fitted μ_eff of run 1a with the dead time off, as read, and corrected with
`dead_time_correct` before the fit.

In [ ]:
r0, r, dt = data["1a  21 kV, no Zr"]
fit_raw = fits["1a  21 kV, no Zr"]
fit_cor = attenuation_fit(
    AL_MM,
    dead_time_correct(r, tau),
    dead_time_correct(r0, tau),
    dt,
    dt,
    bg,
    BG_DT_S,
)
no_dt = DEFAULT.replace(tau_s=0.0)
r0n, rn = al_wheel(21.0, 0.05, dt, params=no_dt)
fit_none = attenuation_fit(AL_MM, rn, r0n, dt, dt, bg, BG_DT_S)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
x = np.linspace(0, 3000, 200)
axes[0].plot(x, x, color="#999", lw=1, label="ideal")
axes[0].plot(x, dead_time(x, tau), color=MODEL, label="τ = 100 µs")
axes[0].set(xlabel="true rate (1/s)", ylabel="observed rate (1/s)")
axes[0].legend(frameon=False)
for f, lab, c in (
    (fit_none, "τ = 0", "#999"),
    (fit_raw, "as read", DATA),
    (fit_cor, "dead-time corrected", ALT),
):
    axes[1].plot(AL_MM, 10 * f.mu_eff_per_mm / RHO_AL, "o-", ms=4, color=c, label=lab)
axes[1].set(xlabel="d (mm Al)", ylabel="μ_eff/ρ (cm²/g)")
axes[1].legend(frameon=False)
fig.tight_layout()

## Things to read correctly

- **μ is an effective value.** The polychromatic beam hardens, so μ depends on the thickness range, U, the filter and
  the dead time. At 21 kV it lies above the monochromatic Mo Kα value (5.11 cm²/g for Al) because the beam is softer
  than Kα; at 35 kV with Zr it lies below.
- **The ± is counting statistics only.** χ²/dof ≫ 1 flags the curvature, not a bad measurement.
- **Dead time is not corrected by the fit.** It matters above a few hundred counts per second.
- **The Pyrex wall changes the spectrum**, hence μ_eff: a 20 % error in its thickness moves μ_eff by 3–8 %, most at the thinnest absorber.